In [1]:
import pyspark 
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *
# Safely fetch the running session or create one if it doesn't exist
spark = SparkSession.builder \
    .appName("test_app") \
    .master("local") \
    .getOrCreate()
    

sc = spark.sparkContext

print("Spark initialized successfully!")

Spark initialized successfully!


In [2]:
data = spark.read.csv(r"C:\Users\Nitro 5\OneDrive\Desktop\Practice datasets\accepted_2007_to_2018Q4.csv",header=True,inferSchema=True,escape='"')
data_bkup = data

In [3]:
shape = data.count(), len(data.columns)
shape 

(2260701, 151)

In [4]:
def count_of_missing_val_v1(df,cn):
    a = df.filter(col(cn).isNull()).count()
    return a 

count_of_missing_val_v1(data,"installment")


33

In [5]:
#Now rather than manually adding each column simultaneously we'll create a more improved version of functions which will give us the missing values and its percentage
# for the column slabs we created.

# def count_of_missing_valv2(df,lst_cl):
#     missing_val={}
#     for i in lst_cl:
#         a = df.filter(col(i).isNull()).count()
#         missing_val[i] = a 
#     return missing_val

# count_of_missing_valv2(data,col_A7)


# def percentage_missing_val(df,lst_cl):
#     missing_val_per={}
#     for i in lst_cl:
#         a = df.filter(col(i).isNull()).count()
#         b = data.count()
#         c = (a/b)*100
#         missing_val_per[i] = c
#     return missing_val_per

   

In [6]:
def percentage_missing_val(df,lst_cl):
    global missing_val_per_above75
    global missing_val_per_below75
    missing_val_per_above75={}
    missing_val_per_below75={}
    for i in lst_cl:
        a = df.filter(col(i).isNull()).count()
        b = df.count()
        c = (a/b)*100
        if c>=75 :
            missing_val_per_above75[i]=c
        else:
            missing_val_per_below75[i]=c
    return ({"missing_val_per_above75":missing_val_per_above75,
            "missing_val_per_below75":missing_val_per_below75})


In [7]:
# here we have created 2 seperate dictionaries with the help of this function
# and we have classified them into 2 groups and going forward we are going to work on them
percentage_missing_val(data,data.columns)

{'missing_val_per_above75': {'member_id': 100.0,
  'desc': 94.42354384768265,
  'mths_since_last_record': 84.11306935326698,
  'annual_inc_joint': 94.66050574578416,
  'dti_joint': 94.66068268205305,
  'verification_status_joint': 94.88079140054346,
  'mths_since_recent_bc_dlq': 77.01151103131285,
  'revol_bal_joint': 95.22183605881538,
  'sec_app_fico_range_low': 95.22179182474817,
  'sec_app_fico_range_high': 95.22179182474817,
  'sec_app_earliest_cr_line': 95.22179182474817,
  'sec_app_inq_last_6mths': 95.22179182474817,
  'sec_app_mort_acc': 95.22179182474817,
  'sec_app_open_acc': 95.22179182474817,
  'sec_app_revol_util': 95.30304980623266,
  'sec_app_open_act_il': 95.22179182474817,
  'sec_app_num_rev_accts': 95.22179182474817,
  'sec_app_chargeoff_within_12_mths': 95.22179182474817,
  'sec_app_collections_12_mths_ex_med': 95.22179182474817,
  'sec_app_mths_since_last_major_derog': 98.41013915595207,
  'hardship_type': 99.51709668815116,
  'hardship_reason': 99.51709668815116,
 

In [8]:
# here we are filtering the columns whose missing value is above 75 as they wont be much of use for analysis purpose

missing_val_above75_col = list(missing_val_per_above75.keys())

# this function represents the list that we have created for dropping the columns and keeping the only useful columns for further analysis.
def columns_tobe_dropped(df,col_lst):
    for i in col_lst:
        df=df.drop(i)
        print("columns dropped",i)
    return df


data = columns_tobe_dropped(data,missing_val_above75_col)
len(data.columns)



columns dropped member_id
columns dropped desc
columns dropped mths_since_last_record
columns dropped annual_inc_joint
columns dropped dti_joint
columns dropped verification_status_joint
columns dropped mths_since_recent_bc_dlq
columns dropped revol_bal_joint
columns dropped sec_app_fico_range_low
columns dropped sec_app_fico_range_high
columns dropped sec_app_earliest_cr_line
columns dropped sec_app_inq_last_6mths
columns dropped sec_app_mort_acc
columns dropped sec_app_open_acc
columns dropped sec_app_revol_util
columns dropped sec_app_open_act_il
columns dropped sec_app_num_rev_accts
columns dropped sec_app_chargeoff_within_12_mths
columns dropped sec_app_collections_12_mths_ex_med
columns dropped sec_app_mths_since_last_major_derog
columns dropped hardship_type
columns dropped hardship_reason
columns dropped hardship_status
columns dropped deferral_term
columns dropped hardship_amount
columns dropped hardship_start_date
columns dropped hardship_end_date
columns dropped payment_plan

111

In [9]:
# As here we can see that the data here is big and the numbe of columns is also 151 its very difficult to study or go through this dataset thats why 
# we'll be creating columns slabs/clusters/portions/groups to make sure that we can see as well as understand the data in better sense.

col_A7 = [i for i in data.columns[0:7]]
col_B7 = [i for i in data.columns[7:15]]
col_C7 = [i for i in data.columns[15:22]]
col_D7 = [i for i in data.columns[22:29]]
col_E7 = [i for i in data.columns[29:36]]
col_F7 = [i for i in data.columns[36:46]]
col_G7 = [i for i in data.columns[46:56]]
col_H7 = [i for i in data.columns[56:66]]
col_I7 = [i for i in data.columns[66:76]]
col_J7 = [i for i in data.columns[76:86]]
col_K7 = [i for i in data.columns[86:96]]
col_L7 = [i for i in data.columns[96:112]]


In [10]:
col_A7

['id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment']

In [11]:
# after the analysis of data removed the values and all the rows from the id columns and the subsequent rows as all the values were nulls and the id column consisted of string values 
data = data.select("*").filter(~col('id').rlike("[A-Za-z]"))

data.count()
shape

(2260701, 151)

In [12]:
# hwew we have successfully cleaned and replaced the values succcessfully for further analysis

data=data.withColumnRenamed("term","term_in_months").withColumn("term_in_months",trim(regexp_replace(col("term_in_months"),"months","")))
data.select("term_in_months").distinct().show()

+--------------+
|term_in_months|
+--------------+
|            60|
|            36|
+--------------+



In [13]:
col_B7

['grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d']

In [14]:
# no issues with grade columnn
data.select("grade").distinct().show()

+-----+
|grade|
+-----+
|    F|
|    E|
|    B|
|    D|
|    C|
|    A|
|    G|
+-----+



In [15]:
# no changes are required as the conditions are met 
data.select("sub_grade").filter(col("sub_grade").rlike("^[A-Z][0-9]")).distinct().show()


+---------+
|sub_grade|
+---------+
|       D5|
|       F2|
|       B4|
|       A2|
|       E4|
|       B2|
|       C3|
|       D1|
|       C4|
|       F1|
|       D3|
|       F5|
|       G2|
|       B1|
|       B3|
|       E5|
|       C5|
|       G3|
|       A4|
|       F4|
+---------+
only showing top 20 rows


In [16]:
# employee title column fixed - capatalized the values and trimmed the spaces.
data= data.withColumn("emp_title",trim(initcap(col("emp_title")))).fillna("N/A(Not Applicable)",subset="emp_title")


In [17]:
data = data.withColumn("emp_length", regexp_replace(col("emp_length"),"[^0-9]"," ")).withColumnRenamed("emp_length","emp_length_in_years")\
.withColumn("emp_length_in_years",trim("emp_length_in_years"))



In [18]:
data.show(5)


+--------+---------+-----------+---------------+--------------+--------+-----------+-----+---------+--------------------+-------------------+--------------+----------+-------------------+--------+-----------+----------+--------------------+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+------------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----

In [19]:
mean_emp_len=data.select("emp_length_in_years").agg(avg("emp_length_in_years")).first()[0]
mean_emp_len=int(mean_emp_len)

In [20]:
data= data.withColumn("emp_length_in_years",col("emp_length_in_years").cast("int"))

In [21]:
data=data.fillna(mean_emp_len,subset=["emp_length_in_years"])


In [22]:
data.select("emp_length_in_years").distinct().show()

+-------------------+
|emp_length_in_years|
+-------------------+
|                  1|
|                  6|
|                  3|
|                  5|
|                  9|
|                  4|
|                  8|
|                  7|
|                 10|
|                  2|
+-------------------+



In [23]:
len(data.columns)

111

In [24]:
data.show(5,False)

+--------+---------+-----------+---------------+--------------+--------+-----------+-----+---------+---------------------------+-------------------+--------------+----------+-------------------+--------+-----------+----------+-----------------------------------------------------------------+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+------------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+---------

In [25]:
data_type=dict(data.dtypes)["annual_inc"]
data_type

'double'

In [26]:
#Created a custom function for gettiing the value for the entered column.

def get_datatype(df,col_name):
    col_dtype=dict(df.dtypes)[col_name]
    return col_dtype



In [27]:
# as we need to fill the null values with the mean we''ll firstly calculate the avg/mean and then imppute those in the maon dataframe

avg_annual_inc=data.select("annual_inc").agg(avg("annual_inc")).first()[0]
int(avg_annual_inc)

77992

In [28]:
data = data.fillna(avg_annual_inc,subset=["annual_inc"])
data.filter(col("annual_inc").isNull()).count()


0

In [29]:
# Fixed the issue_d column datatype ,as it was a string by default converted it into date datatype.

data=data.withColumn("issue_d",to_date(col("issue_d"),"MMM-yyyy"))

get_datatype(data,"issue_d")



'date'

In [30]:
data.select("issue_d").show(10)

+----------+
|   issue_d|
+----------+
|2015-12-01|
|2015-12-01|
|2015-12-01|
|2015-12-01|
|2015-12-01|
|2015-12-01|
|2015-12-01|
|2015-12-01|
|2015-12-01|
|2015-12-01|
+----------+
only showing top 10 rows


In [31]:
col_B7

['grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d']

In [32]:
col_C7

['loan_status',
 'pymnt_plan',
 'url',
 'purpose',
 'title',
 'zip_code',
 'addr_state']

In [33]:
data.select(col_C7).show(5)

+-----------+----------+--------------------+------------------+------------------+--------+----------+
|loan_status|pymnt_plan|                 url|           purpose|             title|zip_code|addr_state|
+-----------+----------+--------------------+------------------+------------------+--------+----------+
| Fully Paid|         n|https://lendingcl...|debt_consolidation|Debt consolidation|   190xx|        PA|
| Fully Paid|         n|https://lendingcl...|    small_business|          Business|   577xx|        SD|
| Fully Paid|         n|https://lendingcl...|  home_improvement|              NULL|   605xx|        IL|
|    Current|         n|https://lendingcl...|debt_consolidation|Debt consolidation|   076xx|        NJ|
| Fully Paid|         n|https://lendingcl...|    major_purchase|    Major purchase|   174xx|        PA|
+-----------+----------+--------------------+------------------+------------------+--------+----------+
only showing top 5 rows


In [34]:
# No changes needed

data.select("loan_status").distinct().show()

+--------------------+
|         loan_status|
+--------------------+
|          Fully Paid|
|             Default|
|     In Grace Period|
|         Charged Off|
|  Late (31-120 days)|
|             Current|
|   Late (16-30 days)|
|Does not meet the...|
|Does not meet the...|
+--------------------+



In [35]:
#No changes needed as all the parameters are verrified.
data.select("pymnt_plan").distinct().count()
data.select("pymnt_plan").distinct().show()
data.filter(col("pymnt_plan").isNull()).count()
data.filter(length(col("pymnt_plan"))>1).count()

+----------+
|pymnt_plan|
+----------+
|         n|
|         y|
+----------+



0

In [36]:
# NO changes needed in url column.
data.select("url").distinct().count()
data.select("url").show(10)
data.filter(col("url").isNull()).count()

+--------------------+
|                 url|
+--------------------+
|https://lendingcl...|
|https://lendingcl...|
|https://lendingcl...|
|https://lendingcl...|
|https://lendingcl...|
|https://lendingcl...|
|https://lendingcl...|
|https://lendingcl...|
|https://lendingcl...|
|https://lendingcl...|
+--------------------+
only showing top 10 rows


0

In [37]:
# No changes needed in purpose column.
data.select("purpose").distinct().show()
data.filter(col("purpose").isNull()).count()

+------------------+
|           purpose|
+------------------+
|           wedding|
|             other|
|    small_business|
|debt_consolidation|
|       credit_card|
|            moving|
|          vacation|
|  renewable_energy|
|             house|
|               car|
|    major_purchase|
|           medical|
|  home_improvement|
|       educational|
+------------------+



0

In [38]:
# title column cleaned and null values imputed with N/A
data.select("title").distinct().count()
data.select("title").distinct().show(10)
data.filter(col("title").isNull()).count()
data=data.fillna("N/A",subset=["title"])

+--------------------+
|               title|
+--------------------+
|      Major purchase|
|             odymeds|
|               Other|
|Credit card refin...|
|         Home buying|
|    Home improvement|
|            Business|
|          Green loan|
|            Vacation|
|    Medical expenses|
+--------------------+
only showing top 10 rows


In [39]:
data.filter(col("title").isNull()).count()

0

In [40]:
# zip code column is cleaned and null is replaced by unavailable.
data.select("zip_code").distinct().count()
# data.filter(col("zip_code").isNull()).show()
data=data.fillna("Unavailable",subset=["zip_code"])
data.filter(col("zip_code").isNull()).count()

0

In [41]:
data.filter(length(col("zip_code"))>5).show()

+--------+---------+-----------+---------------+--------------+--------+-----------+-----+---------+-----------+-------------------+--------------+----------+-------------------+----------+-----------+----------+--------------------+------------------+------------------+-----------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+---------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----------+

In [42]:
# address state column is checked.
data.select("addr_state").distinct().show(10)
# data.filter(col("addr_state").isNull()).count() --- No nulls
data.filter(length(col("addr_state"))>2).count()

+----------+
|addr_state|
+----------+
|        SC|
|        AZ|
|        LA|
|        MN|
|        NJ|
|        DC|
|        OR|
|        VA|
|        RI|
|        WY|
+----------+
only showing top 10 rows


0

In [43]:
col_D7

['dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq']

In [44]:
# performed mean imputaion on dti column for filling up null values.

data.select("dti").distinct().count()
data.select("dti").distinct().show(10)
get_datatype(data,"dti")

mean_dti = data.select("dti").agg(avg("dti")).first()[0]
mean_dti=int(mean_dti)
data=data.fillna(mean_dti,subset=["dti"])
data.filter(col("dti").isNull()).count()

+-----+
|  dti|
+-----+
|17.56|
| 8.51|
|38.61|
| 2.86|
| 15.5|
|19.98|
|23.04|
|37.81|
| 14.9|
|12.32|
+-----+
only showing top 10 rows


0

In [45]:
# performed mean imputation on delinq_2yrs column for filling up nulls 
# data.select("delinq_2yrs").show(5)
# get_datatype(data,"delinq_2yrs")
mean_delinq_2yrs=data.select("delinq_2yrs").agg(avg("delinq_2yrs")).first()[0]
mean_delinq_2yrs=int(mean_delinq_2yrs)

data=data.fillna(mean_delinq_2yrs,subset=["delinq_2yrs"])
data.filter(col("delinq_2yrs").isNull()).count()



0

In [46]:
# 29 nulls 
#
# data.select("earliest_cr_line").distinct().show(5)
data.select("*").filter(col("earliest_cr_line").isNull()).show(5)
# data = data.withColumn("earliest_cr_line",to_date(col("earliest_cr_line"),"MMM-yyyy"))
# data.select("earliest_cr_line").show(5)


+-----+---------+-----------+---------------+--------------+--------+-----------+-----+---------+-------------------+-------------------+--------------+-----------------+-------------------+----------+--------------------+----------+--------------------+-------+--------------------+--------+----------+----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+---------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+----

In [47]:
#  fico_range_low need no changes as it has passed all the test cases.
data.select("fico_range_low").show(5)
data.filter(col("fico_range_low").isNull()).count()
data.filter(col("fico_range_low").rlike("[A-Za-z]")).count()

+--------------+
|fico_range_low|
+--------------+
|         675.0|
|         715.0|
|         695.0|
|         785.0|
|         695.0|
+--------------+
only showing top 5 rows


0

In [48]:

data.select("fico_range_high").show(5)
data.filter(col("fico_range_high").isNull()).count()
data.filter(col("fico_range_high").rlike("[A-Za-z]")).count()

+---------------+
|fico_range_high|
+---------------+
|          679.0|
|          719.0|
|          699.0|
|          789.0|
|          699.0|
+---------------+
only showing top 5 rows


0

In [49]:
# performed mnean imputaion on inqq_last_6mths for imputing the null values 

data.select("inq_last_6mths").show(5)
data.filter(col("inq_last_6mths").isNull()).count()
get_datatype(data,"inq_last_6mths")


+--------------+
|inq_last_6mths|
+--------------+
|           1.0|
|           4.0|
|           0.0|
|           0.0|
|           3.0|
+--------------+
only showing top 5 rows


'double'

In [50]:
mean_inq_last_6mnths = data.select("inq_last_6mths").agg(avg("inq_last_6mths")).first()[0]
mean_inq_last_6mnths=int(mean_inq_last_6mnths)
data=data.fillna(mean_inq_last_6mnths,subset=["inq_last_6mths"])

data.filter(col("inq_last_6mths").isNull()).count()

0

In [51]:
data.select("mths_since_last_delinq").show(5)
get_datatype(data,"mths_since_last_delinq")

+----------------------+
|mths_since_last_delinq|
+----------------------+
|                  30.0|
|                   6.0|
|                  NULL|
|                  NULL|
|                  12.0|
+----------------------+
only showing top 5 rows


'double'

In [52]:
# performed mean imputation on mths_since_last_delinq for imputing the null values.

mean_mths_since_last_delinq=data.select("mths_since_last_delinq").agg(avg("mths_since_last_delinq")).first()[0]
int(mean_mths_since_last_delinq)

34

In [53]:
data=data.fillna(mean_mths_since_last_delinq,subset=["mths_since_last_delinq"])
data.filter(col("mths_since_last_delinq").isNull()).count()

0

In [54]:
def mean_imputation1(df,col_list):
    for i in col_list:
        a=df.select(i).agg(avg(i)).first()[0]
        a=int(a)
        df=df.fillna(a,subset=[i])
        
    return df
    


In [55]:
# from pyspark.ml.feature import Imputer

# for performing mean imputation on list of columns using imputer

# mean_impute2= Imputer(
#     inputCols=[],
#     outputCols=[]
# ).setStrategy("mean")

# df =mean_imppute.fit(df).transform(df)




In [56]:
col_D7

['dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq']

In [57]:
col_E7

['open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp']

In [58]:
# performed mean imputaion on the open accounts column for  imputing the null values,
# mean imputaion was executed with the help of newly created custom funstion.
data.select("open_acc").show(5)
data.filter(col("open_acc").isNull()).count()

+--------+
|open_acc|
+--------+
|     7.0|
|    22.0|
|     6.0|
|    13.0|
|    12.0|
+--------+
only showing top 5 rows


29

In [59]:
data = mean_imputation1(data,["open_acc"])

In [60]:
# mean imputaion is performed on this column for replacing the null values.
data.select("revol_util").show(5)

data = mean_imputation1(data,["revol_util"])
data.filter(col("revol_util").isNull()).count()

+----------+
|revol_util|
+----------+
|      29.7|
|      19.2|
|      56.2|
|      11.6|
|      64.5|
+----------+
only showing top 5 rows


0

In [61]:
data.select(col_E7).show(5)

+--------+-------+---------+----------+---------+-------------------+---------+
|open_acc|pub_rec|revol_bal|revol_util|total_acc|initial_list_status|out_prncp|
+--------+-------+---------+----------+---------+-------------------+---------+
|     7.0|    0.0|   2765.0|      29.7|     13.0|                  w|      0.0|
|    22.0|    0.0|  21470.0|      19.2|     38.0|                  w|      0.0|
|     6.0|    0.0|   7869.0|      56.2|     18.0|                  w|      0.0|
|    13.0|    0.0|   7802.0|      11.6|     17.0|                  w| 15897.65|
|    12.0|    0.0|  21929.0|      64.5|     35.0|                  w|      0.0|
+--------+-------+---------+----------+---------+-------------------+---------+
only showing top 5 rows


In [62]:
# dropped the 29 rows where  the value was null for maintaining data accuracy for further operations.

data.select("*").filter(col("total_acc").isNull()).show(5,False)
data.select(percentile_approx("total_acc",0.5)).show()
data.select(mode("total_acc")).show()
data=data.dropna(subset=["total_acc"])


+-----+---------+-----------+---------------+--------------+--------+-----------+-----+---------+-------------------+-------------------+--------------+-----------------+-------------------+----------+--------------------------------------------------+----------+--------------------------------------------------------------+-------+------------------------------------+--------+----------+----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+---------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+-----

In [63]:
data.select("loan_status").filter(col("total_acc").isNull()).distinct().show(100,False)

+-----------+
|loan_status|
+-----------+
+-----------+



In [64]:
# keeping the initital list status column as it is after running all the test cases.
data.select("initial_list_status").distinct().show(5)
data.filter(col("initial_list_status").isNull()).count()
data.filter(length(col("initial_list_status"))>1).count()

+-------------------+
|initial_list_status|
+-------------------+
|                  f|
|                  w|
+-------------------+



0

In [65]:
# no changes in out principle column after runnning all the necessary test cases.
data.select("out_prncp").show(5)
data.filter(col("out_prncp").isNull()).count()
data.filter(col("out_prncp").rlike("[A-Za-z]")).count()

+---------+
|out_prncp|
+---------+
|      0.0|
|      0.0|
|      0.0|
| 15897.65|
|      0.0|
+---------+
only showing top 5 rows


0

In [66]:
data.select(col_F7).show(15)

+-------------+------------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+
|out_prncp_inv|       total_pymnt|total_pymnt_inv|total_rec_prncp|total_rec_int|total_rec_late_fee|recoveries|collection_recovery_fee|last_pymnt_d|last_pymnt_amnt|
+-------------+------------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+
|          0.0| 4421.723916800001|        4421.72|         3600.0|       821.72|               0.0|       0.0|                    0.0|    Jan-2019|         122.67|
|          0.0|          25679.66|       25679.66|        24700.0|       979.66|               0.0|       0.0|                    0.0|    Jun-2016|         926.35|
|          0.0|22705.924293878397|       22705.92|        20000.0|      2705.92|               0.0|       0.0|                    0.0|    Jun-2017|        15813.3|
|     15897.65| 

In [67]:
# no changes made in out principle inv column after running all the necessary test cases.
data.select("out_prncp_inv").show(5)
data.filter(col("out_prncp_inv").isNull()).count()
data.filter(col("out_prncp_inv").rlike("[A-Za-z]")).count()

+-------------+
|out_prncp_inv|
+-------------+
|          0.0|
|          0.0|
|          0.0|
|     15897.65|
|          0.0|
+-------------+
only showing top 5 rows


0

In [68]:
# no changes made in total payment column after running all the necessary test cases.
data.select("total_pymnt").show(5)
data.filter(col("total_pymnt").isNull()).count()
data.filter(col("total_pymnt").rlike("[A-Za-z]")).count()

+------------------+
|       total_pymnt|
+------------------+
| 4421.723916800001|
|          25679.66|
|22705.924293878397|
|          31464.01|
|           11740.5|
+------------------+
only showing top 5 rows


0

In [69]:
# we'll be dropping the total_paymnt column as the total_paymnt_inv consists of the duplicate records rounded off to 2 digits

data=data.drop("total_pymnt")

In [70]:
# no changes made in total payment invested column after running all the necessary test cases.
data.select("total_pymnt_inv").show(5)
data.filter(col("total_pymnt_inv").isNull()).count()
data.filter(col("total_pymnt_inv").rlike("[A-Za-z]")).count()

+---------------+
|total_pymnt_inv|
+---------------+
|        4421.72|
|       25679.66|
|       22705.92|
|       31464.01|
|        11740.5|
+---------------+
only showing top 5 rows


0

In [71]:
len(data.columns)

110

In [72]:
# No changes required in this column as it has passed all the test cases.
data.select("total_rec_prncp").show(5)
data.filter(col("total_rec_prncp").isNull()).count()
data.filter(col("total_rec_prncp").rlike("[A-Za-z]")).count()

+---------------+
|total_rec_prncp|
+---------------+
|         3600.0|
|        24700.0|
|        20000.0|
|       19102.35|
|        10400.0|
+---------------+
only showing top 5 rows


0

In [73]:
# No changes required in this column as it has passed all the test cases.
data.select("total_rec_int").show(5)
data.filter(col("total_rec_int").isNull()).count()
data.filter(col("total_rec_int").rlike("[A-Za-z]")).count()

+-------------+
|total_rec_int|
+-------------+
|       821.72|
|       979.66|
|      2705.92|
|     12361.66|
|       1340.5|
+-------------+
only showing top 5 rows


0

In [74]:
# No changes required in this column as it has passed all the test cases.
data.select("total_rec_late_fee").show(5)
data.filter(col("total_rec_late_fee").isNull()).count()
data.filter(col("total_rec_late_fee").rlike("[A-Za-z]")).count()
data.filter(col("total_rec_late_fee")==0).count()
data.filter(col("total_rec_late_fee")!=0).count()

+------------------+
|total_rec_late_fee|
+------------------+
|               0.0|
|               0.0|
|               0.0|
|               0.0|
|               0.0|
+------------------+
only showing top 5 rows


87151

In [75]:
data.count()

2260639

In [76]:
# No changes required in this column as it has passed all the test cases.
data.select("recoveries").show(5)
data.filter(col("recoveries").isNull()).count()
data.filter(col("recoveries").rlike("[A-Za-z]")).count()

+----------+
|recoveries|
+----------+
|       0.0|
|       0.0|
|       0.0|
|       0.0|
|       0.0|
+----------+
only showing top 5 rows


0

In [77]:
# # No changes required in this column as it has passed all the test cases.
data.select("collection_recovery_fee").show(5)
data.filter(col("collection_recovery_fee").isNull()).count()
data.filter(col("collection_recovery_fee").rlike("[A-Za-z]")).count()
data.filter(col("collection_recovery_fee")==0).count()
data.filter(col("collection_recovery_fee")!=0).count()

+-----------------------+
|collection_recovery_fee|
+-----------------------+
|                    0.0|
|                    0.0|
|                    0.0|
|                    0.0|
|                    0.0|
+-----------------------+
only showing top 5 rows


177010

In [78]:
data=data.withColumn("last_pymnt_d",to_date(col("last_pymnt_d"),"MMM-yyyy"))



In [79]:
data.show(5)

+--------+---------+-----------+---------------+--------------+--------+-----------+-----+---------+--------------------+-------------------+--------------+----------+-------------------+----------+-----------+----------+--------------------+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----------+----------

In [80]:
# No changes required in this column as it has passed all the test cases.
data.select("last_pymnt_amnt").show(5)
data.filter(col("last_pymnt_amnt").isNull()).count()
data.filter(col("last_pymnt_amnt").rlike("[A-Za-z]")).count()

+---------------+
|last_pymnt_amnt|
+---------------+
|         122.67|
|         926.35|
|        15813.3|
|          829.9|
|       10128.96|
+---------------+
only showing top 5 rows


0

In [81]:
data.select(col_G7).show(5)

+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+--------------+------------+
|next_pymnt_d|last_credit_pull_d|last_fico_range_high|last_fico_range_low|collections_12_mths_ex_med|mths_since_last_major_derog|policy_code|application_type|acc_now_delinq|tot_coll_amt|
+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+--------------+------------+
|        NULL|          Mar-2019|               564.0|              560.0|                       0.0|                       30.0|        1.0|      Individual|           0.0|       722.0|
|        NULL|          Mar-2019|               699.0|              695.0|                       0.0|                       NULL|        1.0|      Individual|           0.0|         0.0|
|        NULL|          Mar-2019|               704.0|           

In [82]:
#  ON HOLD
# data=data.withColumn("next_pymnt_d",to_date(col("next_pymnt_d"),"MMM-yyyy"))
# data.select("next_pymnt_d").show(5)
data.filter(col("next_pymnt_d").isNull()).count()
data.select("*").filter(col("next_pymnt_d").isNull()).show(5)

+--------+---------+-----------+---------------+--------------+--------+-----------+-----+---------+--------------------+-------------------+--------------+----------+-------------------+----------+-----------+----------+--------------------+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----------+----------

In [83]:
data.select("loan_status").filter(col("next_pymnt_d").isNull()).distinct().show(5)

+-----------+
|loan_status|
+-----------+
| Fully Paid|
|Charged Off|
+-----------+



In [84]:
data.select("issue_d").show(5)

+----------+
|   issue_d|
+----------+
|2015-12-01|
|2015-12-01|
|2015-12-01|
|2015-12-01|
|2015-12-01|
+----------+
only showing top 5 rows


In [85]:
# NO changes performed as the column passed all the necessary test cases
data.select("last_pymnt_amnt").show(5)
get_datatype(data,"last_pymnt_amnt")
data.filter(col("last_pymnt_amnt").isNull()).count()
data.filter(col("last_pymnt_amnt").rlike("[a-zA-Z]")).count()


+---------------+
|last_pymnt_amnt|
+---------------+
|         122.67|
|         926.35|
|        15813.3|
|          829.9|
|       10128.96|
+---------------+
only showing top 5 rows


0

In [86]:
col_G7

['next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt']

In [87]:
def change_todate(df,col_list):
    for i in col_list:
        df=df.withColumn(i,to_date(i,"MMM-yyyy"))
    return df

In [88]:
# Added a new column indicating a (1,0) flag on the basis of next payment date column , ffor maintaining data accuracy.
data = change_todate(data,["next_pymnt_d"])
data.select("next_pymnt_d").show(5)
data=data.withColumn("is_active_loan",when(col("next_pymnt_d").isNull(),0).otherwise(1))
data.select("is_active_loan").distinct().show()


+------------+
|next_pymnt_d|
+------------+
|        NULL|
|        NULL|
|        NULL|
|  2019-04-01|
|        NULL|
+------------+
only showing top 5 rows
+--------------+
|is_active_loan|
+--------------+
|             1|
|             0|
+--------------+



In [89]:
# Changed the data type and replaced the null values with the issue date
data.select("last_credit_pull_d").show(5)
data.select("loan_status").filter(col("last_credit_pull_d").isNull()).distinct().show(5,False)
data= data.withColumn("last_credit_pull_d",to_date(col("last_credit_pull_d"),"MMM-yyyy"))
data=data.withColumn("last_credit_pull_d",coalesce(col("last_credit_pull_d"),col("issue_d")))
data.filter(col("last_credit_pull_d").isNull()).count()

+------------------+
|last_credit_pull_d|
+------------------+
|          Mar-2019|
|          Mar-2019|
|          Mar-2019|
|          Mar-2019|
|          Mar-2018|
+------------------+
only showing top 5 rows
+--------------------------------------------------+
|loan_status                                       |
+--------------------------------------------------+
|Fully Paid                                        |
|Charged Off                                       |
|Current                                           |
|Does not meet the credit policy. Status:Fully Paid|
+--------------------------------------------------+



0

In [90]:
# No changes needed as the column has passed all the test cases
data.select("last_fico_range_high").show(5)
data.filter(col("last_fico_range_high").isNull()).count()


+--------------------+
|last_fico_range_high|
+--------------------+
|               564.0|
|               699.0|
|               704.0|
|               679.0|
|               704.0|
+--------------------+
only showing top 5 rows


0

In [91]:
# No changes needed as the column has passed all the test cases
data.select("last_fico_range_low").show(5)
data.filter(col("last_fico_range_low").isNull()).count()


+-------------------+
|last_fico_range_low|
+-------------------+
|              560.0|
|              695.0|
|              700.0|
|              675.0|
|              700.0|
+-------------------+
only showing top 5 rows


0

In [92]:
# Replaced the nulls with 0 foer maintaining the data accuracy.
data.select("collections_12_mths_ex_med").show(5)
data=data.fillna(0,subset=["collections_12_mths_ex_med"])


+--------------------------+
|collections_12_mths_ex_med|
+--------------------------+
|                       0.0|
|                       0.0|
|                       0.0|
|                       0.0|
|                       0.0|
+--------------------------+
only showing top 5 rows


In [93]:
# Replaced the nulls with 0 foer maintaining the data accuracy.
data.select("mths_since_last_major_derog").show(5)
data=data.fillna(0,subset=["mths_since_last_major_derog"])
data.filter(col("mths_since_last_major_derog").isNull()).count()

+---------------------------+
|mths_since_last_major_derog|
+---------------------------+
|                       30.0|
|                       NULL|
|                       NULL|
|                       NULL|
|                       NULL|
+---------------------------+
only showing top 5 rows


0

In [94]:
# No changes needed as the column has passed all the test cases
data.select("policy_code").show(5)
data.filter(col("policy_code").isNull()).count()
get_datatype(data,"policy_code")

+-----------+
|policy_code|
+-----------+
|        1.0|
|        1.0|
|        1.0|
|        1.0|
|        1.0|
+-----------+
only showing top 5 rows


'double'

In [95]:
# No changes needed as the column has passed all the test cases
data.select("application_type").distinct().show()
data.filter(col("application_type").isNull()).count()


+----------------+
|application_type|
+----------------+
|       Joint App|
|      Individual|
+----------------+



0

In [96]:
# No changes needed as the column has passed all the test cases
data.select("acc_now_delinq").show(5)
data.filter(col("acc_now_delinq").isNull()).count()

+--------------+
|acc_now_delinq|
+--------------+
|           0.0|
|           0.0|
|           0.0|
|           0.0|
|           0.0|
+--------------+
only showing top 5 rows


0

In [97]:
# Replaced the nulls with 0 foer maintaining the data accuracy.
data.select("tot_coll_amt").show(5)
# data.filter(col("tot_coll_amt").isNull()).count()
data = data.fillna(0,subset=["tot_coll_amt"])
data.filter(col("tot_coll_amt").isNull()).count()

+------------+
|tot_coll_amt|
+------------+
|       722.0|
|         0.0|
|         0.0|
|         0.0|
|         0.0|
+------------+
only showing top 5 rows


0

In [98]:
col_H7

['tot_cur_bal',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m',
 'mths_since_rcnt_il',
 'total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m']

In [99]:
# as the data in the column policy code is redundant , we'll be dropping that column.
data=data.drop("policy_code")

In [100]:
def zero_imputation(df,col_list):
    for i in col_list:
        df= df.fillna(0,subset=[i])
    return df

In [101]:
data.select("tot_cur_bal").show(5)
data.filter(col("tot_cur_bal").isNull()).count()

+-----------+
|tot_cur_bal|
+-----------+
|   144904.0|
|   204396.0|
|   189699.0|
|   301500.0|
|   331730.0|
+-----------+
only showing top 5 rows


70247

In [102]:

def null_count_per_col(df,col_list):
    nulls_per_col={}
    for i in col_list:
        a=data.filter(col(i).isNull()).count()
        nulls_per_col[i]=a
    return nulls_per_col



In [103]:
null_count_per_col(data,['tot_cur_bal',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m',
 'mths_since_rcnt_il',
 'total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m'])

{'tot_cur_bal': 70247,
 'open_acc_6m': 866101,
 'open_act_il': 866100,
 'open_il_12m': 866100,
 'open_il_24m': 866100,
 'mths_since_rcnt_il': 909895,
 'total_bal_il': 866100,
 'il_util': 1068821,
 'open_rv_12m': 866100,
 'open_rv_24m': 866100}

In [104]:
# Replaced nulls with 0 for maintaining the data accuracy in the columns
data = zero_imputation(data,['tot_cur_bal',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m'])

In [105]:
null_count_per_col(data,['tot_cur_bal',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m'])

{'tot_cur_bal': 0,
 'open_acc_6m': 0,
 'open_act_il': 0,
 'open_il_12m': 0,
 'open_il_24m': 0}

In [106]:
# Replaced nulls with 0 for maintaining the data accuracy in the columns
data.select("mths_since_rcnt_il").show(5)
# null_count_per_col(data,["mths_since_rcnt_il"])
data = zero_imputation(data,["mths_since_rcnt_il"])
null_count_per_col(data,["mths_since_rcnt_il"])

+------------------+
|mths_since_rcnt_il|
+------------------+
|              21.0|
|              19.0|
|              19.0|
|              23.0|
|              14.0|
+------------------+
only showing top 5 rows


{'mths_since_rcnt_il': 0}

In [107]:
data.select('total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m').show(5)


null_count_per_col(data,['total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m'])

+------------+-------+-----------+-----------+
|total_bal_il|il_util|open_rv_12m|open_rv_24m|
+------------+-------+-----------+-----------+
|      4981.0|   36.0|        3.0|        3.0|
|     18005.0|   73.0|        2.0|        3.0|
|     10827.0|   73.0|        0.0|        2.0|
|     12609.0|   70.0|        1.0|        1.0|
|     73839.0|   84.0|        4.0|        7.0|
+------------+-------+-----------+-----------+
only showing top 5 rows


{'total_bal_il': 866100,
 'il_util': 1068821,
 'open_rv_12m': 866100,
 'open_rv_24m': 866100}

In [108]:
data = zero_imputation(data,['total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m'])

null_count_per_col(data,['total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m'])

{'total_bal_il': 0, 'il_util': 0, 'open_rv_12m': 0, 'open_rv_24m': 0}

In [109]:
col_I7

['max_bal_bc',
 'all_util',
 'total_rev_hi_lim',
 'inq_fi',
 'total_cu_tl',
 'inq_last_12m',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util']

In [110]:
data.select("*").filter(col('max_bal_bc').isNull()).show(5)

+--------+---------+-----------+---------------+--------------+--------+-----------+-----+---------+--------------------+-------------------+--------------+----------+-------------------+----------+-----------+----------+--------------------+------------------+--------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----------+------------+-------

In [111]:
null_count_per_col(data,col_I7)

{'max_bal_bc': 866100,
 'all_util': 866319,
 'total_rev_hi_lim': 70247,
 'inq_fi': 866100,
 'total_cu_tl': 866101,
 'inq_last_12m': 866101,
 'acc_open_past_24mths': 50001,
 'avg_cur_bal': 70317,
 'bc_open_to_buy': 74906,
 'bc_util': 76042}

In [112]:
null_count_per_col(data,col_J7)

{'chargeoff_within_12_mths': 116,
 'delinq_amnt': 0,
 'mo_sin_old_il_acct': 139042,
 'mo_sin_old_rev_tl_op': 70248,
 'mo_sin_rcnt_rev_tl_op': 70248,
 'mo_sin_rcnt_tl': 70247,
 'mort_acc': 50001,
 'mths_since_recent_bc': 73383,
 'mths_since_recent_inq': 295406,
 'mths_since_recent_revol_delinq': 1520280}

In [113]:
null_count_per_col(data,col_K7)

{'num_accts_ever_120_pd': 70247,
 'num_actv_bc_tl': 70247,
 'num_actv_rev_tl': 70247,
 'num_bc_sats': 58561,
 'num_bc_tl': 70247,
 'num_il_tl': 70247,
 'num_op_rev_tl': 70247,
 'num_rev_accts': 70248,
 'num_rev_tl_bal_gt_0': 70247,
 'num_sats': 58561}

In [114]:
null_count_per_col(data,col_L7)

{'num_tl_120dpd_2m': 153628,
 'num_tl_30dpd': 70247,
 'num_tl_90g_dpd_24m': 70247,
 'num_tl_op_past_12m': 70247,
 'pct_tl_nvr_dlq': 70402,
 'percent_bc_gt_75': 75350,
 'pub_rec_bankruptcies': 1336,
 'tax_liens': 76,
 'tot_hi_cred_lim': 70247,
 'total_bal_ex_mort': 50001,
 'total_bc_limit': 50001,
 'total_il_high_credit_limit': 70247,
 'hardship_flag': 0,
 'disbursement_method': 0,
 'debt_settlement_flag': 0}

In [115]:
# NO cchanges neeed in this column as it fulfills all the test cases.
data.select("hardship_flag").distinct().show(15)
data.filter(col("hardship_flag").isNull()).count()

+-------------+
|hardship_flag|
+-------------+
|            Y|
|            N|
+-------------+



0

In [116]:
# NO cchanges neeed in this column as it fulfills all the test cases.
data.select("disbursement_method").distinct().show()
data.filter(col("disbursement_method").isNull()).count()

+-------------------+
|disbursement_method|
+-------------------+
|               Cash|
|          DirectPay|
+-------------------+



0

In [117]:
# NO cchanges neeed in this column as it fulfills all the test cases.
data.select("delinq_amnt").distinct().show()
data.filter(col("delinq_amnt").isNull()).count()

+-----------+
|delinq_amnt|
+-----------+
|    13533.0|
|      769.0|
|     1051.0|
|      305.0|
|      184.0|
|      147.0|
|    42904.0|
|     2382.0|
|      169.0|
|      571.0|
|       70.0|
|     8428.0|
|        8.0|
|       67.0|
|      702.0|
|      168.0|
|        0.0|
|      206.0|
|      757.0|
|      249.0|
+-----------+
only showing top 20 rows


0

In [118]:
# Function for for imputing nulls with resulting mode value
def mode_imputaion(df,col_list):
    for i in col_list:
        a=df.select(i).agg(mode(i)).first()[0]
        a=int(a)
        df=df.fillna(a,subset=[i])
    return df

In [119]:
data=mode_imputaion(data,["chargeoff_within_12_mths"])

data.show(5)

+--------+---------+-----------+---------------+--------------+--------+-----------+-----+---------+--------------------+-------------------+--------------+----------+-------------------+----------+-----------+----------+--------------------+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----------+------------+---------

In [120]:
# Performed mode imputation for imputing null values and maintaining the data accuracy.
data.filter(col("chargeoff_within_12_mths").isNull()).count()

0

In [121]:
data.select('max_bal_bc').show(10)
data=data.fillna(0,subset=["max_bal_bc"])
data.filter(col("max_bal_bc").isNull()).count()

+----------+
|max_bal_bc|
+----------+
|     722.0|
|    6472.0|
|    2081.0|
|    6987.0|
|    9702.0|
|    4522.0|
|   13048.0|
|     640.0|
|    2524.0|
|    4725.0|
+----------+
only showing top 10 rows


0

In [136]:
# Performing targeted null imputation across column groups to maintain statistical accuracy

null_impute_list=["max_bal_bc", "total_rev_hi_lim", "avg_cur_bal", "bc_open_to_buy","inq_fi", "total_cu_tl", "inq_last_12m", "acc_open_past_24mths", "mort_acc","num_accts_ever_120_pd", 
                  "num_actv_bc_tl", "num_actv_rev_tl", "num_bc_sats", 'num_bc_tl', "num_il_tl", "num_op_rev_tl", "num_rev_accts", "num_rev_tl_bal_gt_0","num_tl_120dpd_2m"]

data=zero_imputation(data,null_impute_list)



In [123]:
# dropping num sats column as its same as open accounts column 
data.select("num_sats","open_acc").show(10)

+--------+--------+
|num_sats|open_acc|
+--------+--------+
|     7.0|     7.0|
|    22.0|    22.0|
|     6.0|     6.0|
|    13.0|    13.0|
|    12.0|    12.0|
|     5.0|     5.0|
|    12.0|    12.0|
|     8.0|     8.0|
|    14.0|    14.0|
|     8.0|     8.0|
+--------+--------+
only showing top 10 rows


In [142]:
data=data.drop("num_sats","delinq_amnt")
len(data.columns)

108

In [125]:
# List of columns in which the nulls are kept as it is for maintaing the data and statistical accuracy.

cols_without_imputation=["all_util", "bc_util","mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl", "mths_since_recent_bc", "mths_since_recent_inq", "mths_since_recent_revol_delinq"]

In [126]:
# data.write.mode("overwrite").csv(r"C:\Users\Nitro 5\OneDrive\Documents\pyspark_datacleaning\data")

In [143]:
# Pass the data to Pandas to write the CSV natively in Windows
data.limit(10000).toPandas().to_csv(r"C:\Users\Nitro 5\OneDrive\Documents\pyspark_datacleaning\data\cleaned_dataset.csv", index=False)

In [133]:
data.show(5)

+--------+---------+-----------+---------------+--------------+--------+-----------+-----+---------+--------------------+-------------------+--------------+----------+-------------------+----------+-----------+----------+--------------------+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----------+------------+---------